# Imports

In [1]:
import bruce, numpy as np, matplotlib.pyplot as plt, os
from astropy.table import Table, Column
from astropy.stats import sigma_clip
from astropy.time import Time , TimeDelta
from astropy import units as u
import gc, multiprocess
from scipy.stats import median_abs_deviation
from scipy.signal import find_peaks

/Users/sam/anaconda3/lib/python3.9/site-packages/lightkurve/config/__init__.py:119: UserWarning: The default Lightkurve cache directory, used by download(), etc., has been moved to /Users/sam/.lightkurve/cache. Please move all the files in the legacy directory /Users/sam/.lightkurve-cache to the new location and remove the legacy directory. Refer to https://docs.lightkurve.org/reference/config.html#default-cache-directory-migration for more information.
  warnings.warn(


In [2]:
gc.collect()

0

# Load the tics

Data should have the form of (ascii or CSV, your choice). t_zero is in full BJD, width is in days, depth is in normalised flux. If you ue SPOCFIT, if you fit a single transit you will see 3 reported values on the bottom row which are these values for each event (t_zero, width, depth).

tic_id,	t_zero_1,	width_1,	depth_1,	t_zero_2,	width_2,	depth_2,

In [3]:
monos = Table.read('monos.csv')
monos['tic_id'] = monos['TIC ID']
print(monos)

  TIC ID  Linked TIC ID                               TOI ID                                                         DR2 ID                            ... FEROS Other RV   tic_id 
--------- ------------- ----------------------------------------------------------------- ------------------------------------------------------------ ... ----- -------- ---------
 13093071      13093071 FetchError: Service invoked too many times for one day: urlfetch. Error: Service invoked too many times for one day: urlfetch. ...    --       --  13093071
 19265777      19265777 FetchError: Service invoked too many times for one day: urlfetch. Error: Service invoked too many times for one day: urlfetch. ...    --       --  19265777
 21725734      21725734 FetchError: Service invoked too many times for one day: urlfetch. Error: Service invoked too many times for one day: urlfetch. ...    --       --  21725734
 22331531      22331531                                                        Loading... Error: Ser

# Now the main worker function

This function does the following:
-  Load the latest TESS data
-  Flatten the lightcurve
-  Fit the events
-  Calcualte the aliases
-  Plot the permissable aliases
-  Create a report

In [11]:
def func(i):    
    if not os.path.isfile('ngts_data/TIC-{:}.fits'.format(monos['tic_id'][i])) : return
    # Create the output dir (we'll use this as cache for the data too)
    output_dir = os.getcwd() + '/{:}'.format(monos['tic_id'][i])
    os.system('mkdir -p {:}'.format(output_dir))
    
    # Do this if it borked and you want to re-run quickly 
    #if os.path.isfile(output_dir + '/' + 'TIC-{:}_ALIASES.png'.format(duos['tic_id'][i])) : return

    # Now load the TESS data (SPOC, QLP)
    # We are not making our own here like TESSTPF, not yet anyway...
    # for data_type
    #   single_product -> all sectors together
    #   per_sector -> list of per-sector lightcurves
    #   northern_duos -> YEARS 2 and 4, then a list of other sectors 
    #   southern -> YEARS 1 and 3, then a list of other sectors (NOT IMPLEMENTED YET) 
    t, data,data_labels, base_dir =  bruce.ambiguous_period.download_tess_data(monos['tic_id'][i], 
                                                              max_sector=None, 
                                                                   use_ffi=True, 
                                                                   download_dir=None, 
                                                                   bin_length=0.5/24)
    
    # Now flatten the data
    for j, k in zip(data, data_labels):
        # Flatten the data by SG filter, we need an odd kernel legth based on cadence
        j.flatten_data_old(window_width=3, sigmaclip=3, dx_lim=0.1)


#         for seg in bruce.data.find_nights_from_data(j.time, dx_lim=0.2):
#             j.w = np.ones(j.time.shape[0])*np.median(j.flux)

        # Optinally save the data
        j.write_data(output_dir + '/' +'TESS_DATA_{:}.txt'.format(k))
        fig, ax = j.plot_segments(dx_lim=0.5)
        fig.savefig(output_dir + '/' + 'TESS_DATA_{:}.png'.format(k))
        plt.close(fig)
        
        
        
    ########################################################
    # Template match TESS photometry
    ########################################################
    for j in range(len(data)):
        time = np.array(data[j].time, dtype=np.float64)
        flux = np.array(data[j].flux, dtype=np.float64)
        flux_err = np.array(data[j].flux_err, dtype=np.float64)
        normalisation_model = np.array(data[j].w, dtype=np.float64)



        time_trial, delta_L = bruce.template_match.template_match_lightcurve(
                        time,
                        flux,
                        flux_err,
                        normalisation_model,
                        period=monos['FItted Period [day]'][i],
                        radius_1=monos['radius_1 (R1/a)'][i],
                        k=monos['k=R2/R1'][i],
                        incl=np.arccos(monos['radius_1 (R1/a)'][i]*monos['Impact parameter'][i]),
                        e=0.0,
                        w=1.5707963267948966,
                        c=0.7,
                        alpha=0.4,
                        cadence=0,
                        noversample=10,
                        light_3=0.0,
                        ld_law=-2,
                        accurate_tp=1,
                        jitter=0.0,
                        offset=0,
                        time_step=13/60/60/24,
                        time_trial=None,
                    )


        height = 20
        peaks, meta = find_peaks(delta_L, height=height)
        
        # Now plot events
        fig, ax = plt.subplots(1,1)
        ax.plot(time_trial-2457000, delta_L, c='k', lw=1)
        ax.plot(time_trial[peaks]-2457000, delta_L[peaks], 'r+')
        ax.set(xlabel='BTJD', ylabel=r'$\Delta \log \mathcal{L}$', ylim=(0,None))
        ax.axhline(height, c='r', lw=1, ls='--')
        fig.suptitle('TIC-{:} [{:}]'.format(monos['tic_id'][i] , data_labels[j]))
        fig.savefig(output_dir + '/' + 'TESS_DATA_{:}_template_match.png'.format(data_labels[j]))
        plt.close()

        
        # Now plot events
        fig, ax = plt.subplots(2,1, sharex=True)
        ax[0].errorbar(data[j].time-2457000, data[j].flux, yerr=data[j].flux_err, fmt='k.', markersize=1)
        ax[0].plot(data[j].time-2457000, data[j].w, c='r', ls='--')
        ax[0].set(xlabel='BTJD', ylabel='Flux')
        ax[1].plot(time_trial-2457000, delta_L, c='k', lw=1)
        ax[1].plot(time_trial[peaks]-2457000, delta_L[peaks], 'r+')
        ax[1].set(xlabel='BTJD', ylabel=r'$\Delta \log \mathcal{L}$', ylim=(0,None))
        ax[1].axhline(height, c='r', lw=1, ls='--')
        fig.suptitle('TIC-{:} [{:}]'.format(monos['tic_id'][i] , data_labels[j]))
        fig.savefig(output_dir + '/' + 'TESS_DATA_{:}_template_match_full.png'.format(data_labels[j]))
        plt.close()   

        transit_width = bruce.binarystar.transit_width(monos['radius_1 (R1/a)'][i], monos['k=R2/R1'][i], monos['Impact parameter'][i], period=monos['FItted Period [day]'][i])
        
        if len(peaks)>0:
            best_idx = peaks[np.argmax(delta_L[peaks])]
            fig, ax = plt.subplots(2,1, sharex=True)
            ax[0].errorbar(data[j].time-2457000, data[j].flux, yerr=data[j].flux_err, fmt='k.', markersize=1)
            ax[0].plot(data[j].time-2457000, data[j].w, c='r', ls='--')
            ax[1].errorbar(data[j].time-2457000, data[j].flux/data[j].w, yerr=data[j].flux_err/data[j].w, fmt='k.', markersize=1)


            
            t_ = np.linspace(time_trial[best_idx]-transit_width, time_trial[best_idx]+transit_width, 1000)
            model = bruce.binarystar.lc(t_, t_zero = time_trial[best_idx], 
                            period=monos['FItted Period [day]'][i],
                            radius_1=monos['radius_1 (R1/a)'][i],
                            k=monos['k=R2/R1'][i],
                            incl=np.arccos(monos['radius_1 (R1/a)'][i]*monos['Impact parameter'][i]),
                            e=0.0,
                            w=1.5707963267948966,
                            c=0.7,
                            alpha=0.4,
                            cadence=0,
                            noversample=10,
                            light_3=0.0,
                            ld_law=-2,
                            accurate_tp=1,
                            jitter=0.0,
                            offset=0)
            ax[0].plot(t_-2457000, model*np.interp(t_, data[j].time, data[j].w), c='r')
            ax[1].plot(t_-2457000, model, c='r')

            ax[0].set(xlabel='BTJD', ylabel='Flux', xlim=(time_trial[best_idx]-transit_width-2457000, time_trial[best_idx]+transit_width-2457000))
            ax[1].set(xlabel='BTJD', ylabel='Flux')
            fig.suptitle('TIC-{:} [{:}]'.format(monos['tic_id'][i] , data_labels[j]))
            fig.savefig(output_dir + '/' + 'TESS_DATA_{:}_template_match_events.png'.format(data_labels[j]))
            plt.close()
    
    ########################################################
    # CHECK NGTS PHOTOMETRY
    ########################################################
    ngts_data = []
    ngts_data_labels = []
    if os.path.isfile('ngts_data/TIC-{:}.fits'.format(monos['tic_id'][i])):
        ngts = Table.read('ngts_data/TIC-{:}.fits'.format(monos['tic_id'][i]), hdu=4)
        ngts.sort('BJD')
        mask = (ngts['SIGMA_XS']<0.02) & ~np.isnan(np.array(ngts['FLUX_SYSREM_ALIGNED'], dtype=np.float64)) &\
                     ~np.isnan(np.array(ngts['FLUX_ERR'], dtype=np.float64)) &\
                     ~np.isinf(np.array(ngts['FLUX_SYSREM_ALIGNED'], dtype=np.float64)) &\
                     ~np.isinf(np.array(ngts['FLUX_ERR'], dtype=np.float64))
        ngts = ngts[mask]
        
        # Now lets bin
        t_bin, f_bin, fe_bin, c = bruce.data.bin_data(np.array(ngts['BJD'], dtype=np.float64),
                                                  np.array(ngts['FLUX_SYSREM_ALIGNED'], dtype=np.float64),
                                                  0.5/24/3)

        t_bin, f_bin, fe_bin = t_bin[c>10], f_bin[c>10], fe_bin[c>10]
        t_bin, f_bin, fe_bin = t_bin[fe_bin>0], f_bin[fe_bin>0], fe_bin[fe_bin>0]
        N = np.nanmedian(f_bin)
        f_bin = f_bin/N
        fe_bin = fe_bin/N
        
        if len(t_bin)>3 : 
            
            #######################################################################################
            #    OK, there are two ways of doing this based on data volume
            # We could 
            # 1. Split each night into its own LC so we can see how ruling out aliases
            #    progresses. This can be cumbersome and make large plots.
            # 2. Treat the NGTS LC as one, but if wee see a transit we will have 
            #    to do a retrospective calculation to see what aliases is was compatible with.
            #    It also might bork the delta_L calculation if a transit is seen but later excluded.
            #    Furthermore, if we only have a short amount of in-transit data, we have to be
            #    careful about how we detrend.
            #######################################################################################

            
#             #######################################################################################
#             # Aproach 1
#             #######################################################################################

#             # Now create the data products 
#             for seg in bruce.data.find_nights_from_data(t_bin, dx_lim=0.2): 
#                 # Now check the phase is consistent with one of the aliases

#                 # Now get the phases of all aliases
#                 phases = np.zeros((p.aliases.shape[0], len(np.array(t_bin, dtype=np.float64))))
#                 for j in range(len(phases)):
#                     phases[j] = bruce.data.phase_times(t_bin, p.events[0].de_get_epoch(), p.max_period/p.aliases[j], phase_offset=0.2)
#                 phase_widths = p.events[0].de_transit_width() / (p.max_period/p.aliases)

#                 # Now mask the data which doesent fall within 1 width of any alias
#                 useful_data = np.abs(phases) < (phase_widths[:,np.newaxis]/2) # Make for a nice plot

#                 # If so, lets make a data object for the night 
#                 if useful_data.any():
#                     # Now create the time series object
#                     d = bruce.ambiguous_period.photometry_time_series(t_bin[seg], f_bin[seg], fe_bin[seg])
#                     d.w = np.ones(len(d.time))

#                     # Now normalise to the median of the top 20%
#                     try:
#                         quarter_edges = np.linspace(np.min(d.time)-1e-3,np.max(d.time)+1e-3,5)
#                         dig = np.digitize(d.time, quarter_edges, right=True)
#                         medians = np.array([np.nanmedian(d.flux[dig==j]) for j in range(1,len(quarter_edges))])
#                         constant = np.nanmax(medians)
#                     except :  constant = np.nanmedian(d.flux)


#                     d.flux = d.flux / constant
#                     d.flux_err = d.flux_err / constant
#                     ngts_data.append(d)

#                     ngts_data_labels.append(Time(d.time[0], format='jd').datetime.strftime('%Y-%m-%d'))            
#                     print(Time(d.time[0], format='jd').datetime.strftime('%Y-%m-%d'))


            #######################################################################################
            # Aproach 2
            #######################################################################################
            # Now create the time series object
            d = bruce.ambiguous_period.photometry_time_series(t_bin, f_bin, fe_bin)
            d.w = np.ones(len(d.time))
            
            # We will do a crude variability removal
            # BE CAREFUL, only a small amount of in-transit data will 
            for seg in bruce.data.find_nights_from_data(d.time, dx_lim=0.2): 
                d.w[seg] = np.nanmedian(d.flux[seg])*np.ones(seg.shape[0])
            ngts_data.append(d)
            ngts_data_labels.append('NGTS data')
            
            # Now template match 
            time_trial, delta_L = bruce.template_match.template_match_lightcurve(
                            d.time,
                            d.flux,
                            d.flux_err,
                            d.w,
                            period=monos['FItted Period [day]'][i],
                            radius_1=monos['radius_1 (R1/a)'][i],
                            k=monos['k=R2/R1'][i],
                            incl=np.arccos(monos['radius_1 (R1/a)'][i]*monos['Impact parameter'][i]),
                            e=0.0,
                            w=1.5707963267948966,
                            c=0.7,
                            alpha=0.4,
                            cadence=0,
                            noversample=10,
                            light_3=0.0,
                            ld_law=-2,
                            accurate_tp=1,
                            jitter=0.0,
                            offset=0,
                            time_step=13/60/60/24,
                            time_trial=None,
                        )


            height = 5
            peaks, meta = find_peaks(delta_L, height=height)
            print(peaks)
            # Now plot events
            fig, ax = plt.subplots(1,1)
            ax.plot(time_trial-2457000, delta_L, c='k', lw=1)
            ax.plot(time_trial[peaks]-2457000, delta_L[peaks], 'r+')
            ax.set(xlabel='BTJD', ylabel=r'$\Delta \log \mathcal{L}$', ylim=(0,None))
            ax.axhline(height, c='r', lw=1, ls='--')
            fig.suptitle('TIC-{:} [NGTS data]'.format(monos['tic_id'][i]))
            fig.savefig(output_dir + '/' + 'NGTS_DATA_template_match.png')
            plt.close()


            # Now plot events
            fig, ax = plt.subplots(2,1, sharex=True)
            ax[0].errorbar(d.time-2457000, d.flux, yerr=d.flux_err, fmt='k.', markersize=1)
            ax[0].plot(d.time-2457000, d.w, c='r', ls='--')
            ax[0].set(xlabel='BTJD', ylabel='Flux')
            ax[1].plot(time_trial-2457000, delta_L, c='k', lw=1)
            ax[1].plot(time_trial[peaks]-2457000, delta_L[peaks], 'r+')
            ax[1].set(xlabel='BTJD', ylabel=r'$\Delta \log \mathcal{L}$', ylim=(0,None))
            ax[1].axhline(height, c='r', lw=1, ls='--')
            fig.suptitle('TIC-{:} [NGTS data]'.format(monos['tic_id'][i]))
            fig.savefig(output_dir + '/' + 'NGTS_DATA_{:}_template_match_full.png'.format(data_labels[j]))
            plt.close()     
            
            if len(peaks)>0:
                ncols = 2
                nrows= int(np.ceil(len(peaks)/ncols))
                fig, ax = plt.subplots(ncols=ncols, nrows=nrows, figsize = (10,3 + nrows))
                ax = np.array(ax).flatten()
                for j in range(len(peaks)):
                    data_mask = (d.time>(time_trial[peaks[j]]-transit_width)) & (d.time<(time_trial[peaks[j]]+transit_width))
                    print('SUM = ' + str(data_mask.sum()))
                    ax[j].errorbar(d.time[data_mask]-time_trial[peaks[j]], d.flux[data_mask], yerr=d.flux_err[data_mask],fmt='k.', markersize=1)


                    t_ = np.linspace(time_trial[peaks[j]]-transit_width, time_trial[peaks[j]]+transit_width, 1000)
                    model = bruce.binarystar.lc(t_, t_zero = time_trial[peaks[j]], 
                                    period=monos['FItted Period [day]'][i],
                                    radius_1=monos['radius_1 (R1/a)'][i],
                                    k=monos['k=R2/R1'][i],
                                    incl=np.arccos(monos['radius_1 (R1/a)'][i]*monos['Impact parameter'][i]),
                                    e=0.0,
                                    w=1.5707963267948966,
                                    c=0.7,
                                    alpha=0.4,
                                    cadence=0,
                                    noversample=10,
                                    light_3=0.0,
                                    ld_law=-2,
                                    accurate_tp=1,
                                    jitter=0.0,
                                    offset=0)

                    ax[j].plot(t_-time_trial[peaks[j]], model, c='r')
                    ax[j].set(title = 'Peak {:}'.format(j+1), xlabel='BTJD - {:}'.format(time_trial[peaks[j]]-2457000), ylabel='Flux')
                fig.suptitle('TIC-{:} [NGTS data]'.format(monos['tic_id'][i]))
                fig.savefig(output_dir + '/' + 'TIC-{:}_NGTS_DATA_template_match_events.png'.format(monos['tic_id'][i]))
                plt.close()

                

    plt.close()
    gc.collect()


# import multiprocess
# with multiprocess.Pool(6) as p : p.map(func, range(len(monos)))
    


            
            
# f = open('failed.txt', 'w+')
# f.close()
# for i in range(len(duos))[5:]: 
#     try : func(i)
#     except: 
#         with open('failed.txt', 'a+') as f:
#             f.write('{:}\n'.format(duos['tic_id'][i]))

            
# print(np.argwhere(duos['tic_id']==126565211)[0][0])   
func( np.argwhere(monos['tic_id']==123570723)[0][0] )

#for i in range(len(monos)) : func(i)
# func(1)


📂 Download directory: /var/folders/fs/c4gpqhmx5tbcbsv_cbs0yhn80000gn/T/tmprj43rz_h

Querying TIC 123570723 from MAST...



✅ Download summary:
Sector SPOC FFI QLP LC TESS-SPOC LC SPOC LC    Source                                                                File                                                            
------ -------- ------ ------------ ------- ------------ ----------------------------------------------------------------------------------------------------------------------------
     6       --     --            X      -- TESS-SPOC LC /var/folders/fs/c4gpqhmx5tbcbsv_cbs0yhn80000gn/T/tmprj43rz_h/hlsp_tess-spoc_tess_phot_0000000123570723-s0006_tess_v1_lc.fits
    33       --     --            X      -- TESS-SPOC LC /var/folders/fs/c4gpqhmx5tbcbsv_cbs0yhn80000gn/T/tmprj43rz_h/hlsp_tess-spoc_tess_phot_0000000123570723-s0033_tess_v1_lc.fits
    87       --      X           --      --       QLP LC      /var/folders/fs/c4gpqhmx5tbcbsv_cbs0yhn80000gn/T/tmprj43rz_h/hlsp_qlp_tess_ffi_s0087-0000000123570723_tess_v01_llc.fits

Temporary data stored in: /var/folders/fs/c4gpqhmx5tbcbsv_cbs0yhn800

W :  144
W :  144
W :  144
W :  144
W :  144
W :  144
[  147  7201  9865 19814 24797 35111 40077 52676 60500]
SUM = 9
SUM = 4
SUM = 10
SUM = 11
SUM = 11
SUM = 12
SUM = 13
SUM = 14
SUM = 9


In [12]:
x=1

# Now look for duos

In [10]:
import glob
os.system('mkdir -p duos; rm duos/*')
for i in range(len(monos)):
    files = glob.glob('{:}/*_events.png'.format(monos['tic_id'][i]))
    if len(files)>1:
        print(monos['tic_id'][i])

21725734
22331531
36541001
65910228
67212165
80256205
93330962
101944220
122301267
123570723
130718055
138712345
153723036
156068960
178154917
184534291
186072225
201352685
219579252
224299778
229159618
235039711
237981801
254302997
265371495
303050301
310420973
348063710
350157223
364175700
394354824
455238504
457351794


rm: cannot remove 'duos/*': No such file or directory
